In [1]:
import requests
import pandas as pd
import numpy as np
from openai import OpenAI

In [2]:
# =========================================================
# 1) CONFIGURAÇÃO DA LLM
# =========================================================

# Ajuste conforme seu ambiente
# Exemplo:
# - Cloudera AI Inference compatível com OpenAI
# - outro endpoint compatível
BASE_URL = "https://hol-cai-infe-services.hol-spag.z30z-14kp.cloudera.site/namespaces/serving-default/endpoints/llama-31-8b/v1"
API_KEY = "eyJraWQiOiIzYzhlNzA3OTEyZmI0NTA1ODE3NzE3YzMyOTU4MmQwMTFjYjlmNTAwIiwidHlwIjoiSldUIiwiYWxnIjoiUlMyNTYifQ.eyJzdWIiOiJ2Y29uZGUiLCJhdWQiOiJodHRwczovL2RlLnozMHotMTRrcC5jbG91ZGVyYS5zaXRlIiwiaXNzIjoiaHR0cHM6Ly9jb25zb2xlYXV0aC5jZHAuY2xvdWRlcmEuY29tL2Y4ZTE2ZDJkLTU3ZTYtNDNiYy04ZTUwLTc1YTMyZjc3ODJhZCIsImdyb3VwcyI6ImNkcF9maWVsZF9tYXJrZXRpbmdfYW1lciBob2wtcGljcGF5LWF3LWNkcC1hZG1pbi1ncm91cCBob2wtc3BhZ3Vhcy1pbnN0cnVjdG9yIF9jX21sX2FkbWluc181ZDgzNzY1ZiBfY19uaWZpX3JlZ2lzdHJ5X2FkbWluc18xMzFlN2JhMiBfY19oYmFzZV9hZG1pbnNfMTMxZTdiYTIgX2NfbWxfYWRtaW5zXzEzMWU3YmEyIF9jX2tub3hfYWRtaW5zXzEzMWU3YmEyIF9jX2Vudl9hc3NpZ25lZXNfMTMxZTdiYTIgX2NfbmlmaV9hZG1pbnNfMTMxZTdiYTIgX2NfZWZtX2FkbWluc18xMzFlN2JhMiBfY19tbF91c2Vyc18xMzFlN2JhMiBfY196ZXBwZWxpbl9hZG1pbnNfMTMxZTdiYTIgX2NfZGZfcHJvamVjdF9tZW1iZXJfNWFlMDVmN2UgX2NfcmFuZ2VyX2FkbWluc18xMzFlN2JhMiBfY19jbV9hZG1pbnNfMTMxZTdiYTIiLCJleHAiOjE3NzQyODE3OTMsInR5cGUiOiJ1c2VyIiwiZ2l2ZW5fbmFtZSI6IlZpdG9yIiwiaWF0IjoxNzc0Mjc4MTkzLCJmYW1pbHlfbmFtZSI6IkNvbmRlIiwiZW1haWwiOiJ2Y29uZGVAY2xvdWRlcmEuY29tIn0.c8hsBzvPKQvzH0CEjRtxX2xsgioMxIz7wf8Qkx8meDHqOk-NQx5mUfW7qjxrnGxHj3JwIRBdSLo802HSlSxyKxZAvOMJ9Zampckz5SYGWaWrFahXzYQpsQzntOfXYpm8g-39kwvA93C5dRR8pTdSIsFWW-RJWz2GWI3NK8r7U8gf6IAtEYpx02DeayiSJyO1qRJVWMYmdZZoJWLD0vsgs-3xxlxjaFSu8_rLoRd8rLDa6v9jGChbbhGCQx5oteqB3yvP5JQwaV0vl7yH5oq09vTBQIxxvurkH1XQ0UoyCOapPwpOvluifVGJO9dwaqO44QKagWhUiTdOPnLHcKdm5w"
MODEL_NAME = "meta/llama-3.1-8b-instruct"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

In [3]:
# =========================================================
# 2) COLETA E MODELO DE RISCO
# =========================================================

def get_rain_data(hours=6):
    url = "https://apps.spaguas.sp.gov.br/sibh/api/v2/measurements/now"
    params = {
        "station_type_id": 2,
        "hours": hours,
        "show_all": "true",
        "serializer": "complete",
        "public": "true"
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get("measurements", [])


def get_level_data():
    url = "https://apps.spaguas.sp.gov.br/sibh/api/v2/measurements/now_flu"
    params = [
        ("references[]", "extravasation"),
        ("references[]", "emergency"),
        ("references[]", "alert"),
        ("references[]", "attention"),
        ("with_one_ref", "true")
    ]
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    if isinstance(data, dict):
        for key in ["measurements", "data", "results"]:
            if key in data and isinstance(data[key], list):
                return data[key]
    if isinstance(data, list):
        return data
    return []


def prepare_rain_df(rain_data):
    df = pd.json_normalize(rain_data)

    for col in ["station_name", "city", "value"]:
        if col not in df.columns:
            df[col] = None

    df["rain_value"] = pd.to_numeric(df["value"], errors="coerce")
    keep_cols = [c for c in ["station_name", "city", "rain_value", "max_value", "max_date"] if c in df.columns]
    return df[keep_cols].copy()


def prepare_level_df(level_data):
    df = pd.json_normalize(level_data)

    for col in ["station_name", "city", "value", "attention", "alert", "emergency", "extravasation"]:
        if col not in df.columns:
            df[col] = None

    df["level_value"] = pd.to_numeric(df["value"], errors="coerce")

    for ref_col in ["attention", "alert", "emergency", "extravasation"]:
        df[ref_col] = pd.to_numeric(df[ref_col], errors="coerce")

    keep_cols = [c for c in ["station_name", "city", "level_value", "attention", "alert", "emergency", "extravasation"] if c in df.columns]
    return df[keep_cols].copy()


def calculate_level_ratio(row):
    level_value = row.get("level_value", np.nan)
    if pd.isna(level_value):
        return 0.0

    refs = ["attention", "alert", "emergency", "extravasation"]
    score = 0.0

    for ref in refs:
        ref_value = row.get(ref, np.nan)
        if not pd.isna(ref_value) and ref_value != 0:
            score = max(score, level_value / ref_value)

    return min(score, 1.5)


def classify_risk(score):
    if score >= 0.95:
        return "CRÍTICO"
    elif score >= 0.75:
        return "ALTO"
    elif score >= 0.45:
        return "MÉDIO"
    return "BAIXO"


def build_risk_model(hours=6):
    rain_df = prepare_rain_df(get_rain_data(hours))
    level_df = prepare_level_df(get_level_data())

    merged = pd.merge(rain_df, level_df, on=["station_name", "city"], how="outer")

    merged["rain_value"] = merged["rain_value"].fillna(0)
    merged["level_value"] = merged["level_value"].fillna(0)

    max_rain = merged["rain_value"].max() if len(merged) > 0 else 0
    merged["rain_score"] = merged["rain_value"] / max_rain if max_rain > 0 else 0

    merged["level_score"] = merged.apply(calculate_level_ratio, axis=1)
    merged["risk_score"] = merged["rain_score"] * 0.4 + merged["level_score"] * 0.6
    merged["risk_class"] = merged["risk_score"].apply(classify_risk)

    merged = merged.sort_values("risk_score", ascending=False)
    return merged


In [4]:
# =========================================================
# 3) MONTAGEM DO CONTEXTO PARA A LLM
# =========================================================

def build_llm_context(df, top_n=10):
    """
    Cria um contexto curto e legível para a LLM.
    """
    top_df = df.head(top_n).copy()

    lines = []
    lines.append("Resumo das estações mais críticas:\n")

    for _, row in top_df.iterrows():
        station = row.get("station_name", "N/D")
        city = row.get("city", "N/D")
        rain = row.get("rain_value", 0)
        level = row.get("level_value", 0)
        risk_score = row.get("risk_score", 0)
        risk_class = row.get("risk_class", "N/D")
        attention = row.get("attention", None)
        alert = row.get("alert", None)
        emergency = row.get("emergency", None)
        extravasation = row.get("extravasation", None)

        lines.append(
            f"- Estação: {station} | Cidade: {city} | "
            f"Chuva: {rain:.2f} | Nível: {level:.2f} | "
            f"Score: {risk_score:.2f} | Classe: {risk_class} | "
            f"Atenção: {attention} | Alerta: {alert} | "
            f"Emergência: {emergency} | Extravasamento: {extravasation}"
        )

    return "\n".join(lines)


In [5]:
# =========================================================
# 4) CHAMADA DA LLM
# =========================================================

def explain_events_with_llm(df):
    """
    Envia o contexto para a LLM e pede uma explicação do cenário.
    """
    context = build_llm_context(df, top_n=10)

    prompt = f"""
Você é um analista hidrológico.

Com base apenas nos dados abaixo:
- explique o cenário atual de forma simples;
- destaque estações críticas;
- diga por que elas parecem preocupantes;
- aponte diferenças entre chuva alta e nível alto;
- não invente informação fora do contexto.

Contexto:
{context}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "Você responde de forma objetiva, técnica e simples."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


In [6]:
# =========================================================
# 5) EXECUÇÃO
# =========================================================

if __name__ == "__main__":
    df_risk = build_risk_model(hours=6)

    explanation = explain_events_with_llm(df_risk)

    print("\n=== EXPLICAÇÃO GERADA PELA LLM ===\n")
    print(explanation)

    with open("explicacao_eventos_llm.txt", "w", encoding="utf-8") as f:
        f.write(explanation)

    print("\nArquivo salvo: explicacao_eventos_llm.txt")


=== EXPLICAÇÃO GERADA PELA LLM ===

**Cenário Atual:**

O cenário atual é de alerta hidrológico em várias estações da bacia hidrográfica. As estações estão registrando níveis de água altos e alguns sinais de alerta, mas sem extravasamento. Isso indica que a situação é preocupante, mas ainda não é uma emergência.

**Estações Críticas:**

As estações críticas são:

1. **Rancharia (PCH Pari Barramento)**: Nível de água: 37218.00, Score: 0.90, Classe: ALTO. A estação está registrando um nível de água extremamente alto, o que pode indicar uma situação de emergência.
2. **Auriflama (UHE Ilha Solteira Fazenda Palmeirinha)**: Nível de água: 381.00, Score: 0.90, Classe: ALTO. A estação está registrando um nível de água alto, o que pode indicar uma situação de alerta.
3. **São José dos Campos (UHE Jaguari Fazenda Santana)**: Nível de água: 1545.00, Score: 0.90, Classe: ALTO. A estação está registrando um nível de água alto, o que pode indicar uma situação de alerta.
4. **São Joaquim da Barra (P